# Tractogram to Zarr Vectors

Convert a TrackVis `.trk` tractogram into a `.zv` store, build a streamline
pyramid, and inspect the result:

1. Write a synthetic `.trk` file (or use your own)
2. Convert it with `ingest_trk`
3. Build a pyramid with Douglas–Peucker simplification
4. Inspect metadata, headers and the on-disk layout
5. Validate, read back and export a coarse level to TRK

Needs `nibabel` (the `[trk]` extra).

In [1]:
import os
import tempfile
from pathlib import Path

import nibabel as nib
import numpy as np
import zarr_vectors as zv

_tmpdir = tempfile.mkdtemp(prefix="zv_examples_")
TRK_PATH = os.path.join(_tmpdir, "tracts.trk")
STORE = os.path.join(_tmpdir, "tracts.zv")
print("Working directory:", _tmpdir)

Working directory: /tmp/zv_examples_9fsfsr4c


## 1 · Write a synthetic tractogram

2,000 smooth streamlines with 0.5 mm steps inside a 180 × 216 × 180 mm
reference volume (2 mm voxels), each with a per-point `fa` and a
per-streamline `weight`. nibabel takes the points in RAS mm and writes them
to the file in voxmm.

To use a real file instead, set `TRK_PATH` to it and skip this cell.

In [2]:
rng = np.random.default_rng(42)
N = 2_000

dims = np.array([90, 108, 90])
voxel_size = np.array([2.0, 2.0, 2.0])
extent = dims * voxel_size                                  # mm
vox_to_ras = np.diag([2.0, 2.0, 2.0, 1.0])
vox_to_ras[:3, 3] = -extent / 2                             # centre the volume on the origin

def random_streamline(n_points):
    direction = rng.normal(size=3)
    direction /= np.linalg.norm(direction)
    points = [rng.uniform(0.25 * extent, 0.75 * extent)]
    for _ in range(n_points - 1):
        direction += rng.normal(0, 0.1, 3)
        direction /= np.linalg.norm(direction)
        points.append(np.clip(points[-1] + 0.5 * direction, 0, extent - 1e-3))
    return np.array(points) - extent / 2                    # RAS mm

streamlines = [random_streamline(int(rng.integers(60, 160))) for _ in range(N)]
fa = [rng.uniform(0.1, 0.9, (len(s), 1)).astype(np.float32) for s in streamlines]
weight = rng.uniform(0, 1, (N, 1)).astype(np.float32)

tractogram = nib.streamlines.Tractogram(
    streamlines,
    data_per_point={"fa": fa},
    data_per_streamline={"weight": weight},
    affine_to_rasmm=np.eye(4),
)
header = {
    "dimensions": dims.astype(np.int16),
    "voxel_sizes": voxel_size.astype(np.float32),
    "voxel_to_rasmm": vox_to_ras,
    "voxel_order": "RAS",
}
nib.streamlines.save(tractogram, TRK_PATH, header=header)

print(f"{N:,} streamlines, {sum(len(s) for s in streamlines):,} points")
print(f"{os.path.getsize(TRK_PATH) / 1e6:.1f} MB written to {Path(TRK_PATH).name}")

2,000 streamlines, 222,922 points
3.6 MB written to tracts.trk


## 2 · Convert

`ingest_trk` stores positions in RAS mm, `fa` as a vertex attribute and
`weight` as an object attribute, and keeps the TRK header under
`headers/trk` for export. `compute_length=True` adds each streamline's length
in mm. The chunk and bin shapes are in mm.

The command-line equivalent of this cell and the next is:

```bash
zvtools convert tracts.trk tracts.zv --chunk-shape 32,32,32 --bin-shape 4,4,4 \
    --coarsen 1,1,1 --sparsity 2,2,2 --rdp-tolerance 0.5,1,2 --chunk-scale 2,2,2
```

In [3]:
from zarr_vectors_tools.convert.ingest.trk import ingest_trk

summary = ingest_trk(
    TRK_PATH,
    STORE,
    chunk_shape=(32.0, 32.0, 32.0),
    bin_shape=(4.0, 4.0, 4.0),
    compute_length=True,
)
print(summary)

{'polyline_count': 2000, 'vertex_count': 222922, 'chunk_count': 207, 'cross_chunk_link_count': 5156, 'group_count': 0}


## 3 · Build the pyramid

Each coarser level keeps half the streamlines of the level below
(`sparsity` 2), simplifies each one with Douglas–Peucker so it strays at most
0.5, 1 and 2 mm from the level below (`rdp_tolerances`), and doubles the chunk
edge (`chunk_scale_factors`) so that every level can keep simplifying. With
`rdp_tolerances` given, the coarsen factor only sets the number of levels, so
it is 1. `sparsity_seed` makes the random selection repeatable.

In [4]:
from zarr_vectors_tools.multiresolution.coarsen import build_pyramid

info = build_pyramid(
    STORE,
    factors=[(1, 2), (1, 2), (1, 2)],          # (coarsen, sparsity) per level
    chunk_scale_factors=[2, 2, 2],
    rdp_tolerances=[0.5, 1.0, 2.0],            # mm
    sparsity_seed=0,
)
print(f"levels created: {info['levels_created']}  (method: {info['method']})")

ds = zv.open(STORE)
for lv in ds.levels:
    level = ds.level(lv)
    print(f"  Level {lv}: {level.objects.count:>5,} streamlines  "
          f"{level.vertex_count:>8,} vertices  "
          f"({level.vertex_count / level.objects.count:5.1f} per streamline)  "
          f"chunk {level.scale[0]:.0f} mm")

levels created: 3  (method: polyline_rdp)
  Level 0: 2,000 streamlines   222,922 vertices  (111.5 per streamline)  chunk 32 mm
  Level 1: 1,000 streamlines    12,032 vertices  ( 12.0 per streamline)  chunk 64 mm
  Level 2:   500 streamlines     4,062 vertices  (  8.1 per streamline)  chunk 128 mm
  Level 3:   250 streamlines     1,541 vertices  (  6.2 per streamline)  chunk 256 mm


## 4 · Inspect the store

### 4a · Metadata and headers

In [5]:
print(ds)
print(f"format version : {'.'.join(map(str, ds.format_version))}")
print(f"kinds          : {ds.kinds}")
print(f"bounds (mm)    : {ds.bounds[0].round(1)} → {ds.bounds[1].round(1)}")
lv0 = ds.level(0)
print(f"vertex attrs   : {lv0.attribute_names('vertex')}")
print(f"object attrs   : {lv0.attribute_names('object')}")
print(f"headers        : {ds.headers}")

Dataset('file:///tmp/zv_examples_9fsfsr4c/tracts.zv', streamline, levels=[0, 1, 2, 3])
format version : 0.9.3
kinds          : ('streamline',)
bounds (mm)    : [ -90. -108.  -90.] → [ 90. 108.  90.]
vertex attrs   : ('fa',)
object attrs   : ('length', 'vertex_count', 'weight')
headers        : HeaderRegistry(formats=['trk'])


### 4b · Layout on disk

Each numbered directory is a level. Each per-chunk array keeps one file per
non-empty chunk under `c/`, so the coarser levels, with larger chunks, have
fewer files. A streamline store keeps only its chunk-crossing steps as links:
`links/0/<offset>` holds the steps into the neighbouring chunk at that offset.
`fragment_attributes/segment_id` records which streamline each chunk run
belongs to; the pyramid builder reads it.

In [6]:
def describe(path, max_depth=3, _depth=0, prefix=""):
    """List directories to max_depth; summarise each array's chunk files."""
    for entry in sorted(Path(path).iterdir()):
        if not entry.is_dir():
            continue
        chunk_dir = entry / "c"
        if chunk_dir.is_dir():
            files = [f for f in chunk_dir.rglob("*") if f.is_file()]
            size = sum(f.stat().st_size for f in files)
            print(f"{prefix}{entry.name}/  {len(files):>4} chunk files, {size / 1024:7.1f} KiB")
        else:
            print(f"{prefix}{entry.name}/")
            if _depth < max_depth:
                describe(entry, max_depth, _depth + 1, prefix + "    ")

print(Path(STORE).name)
describe(STORE)

tracts.zv
0/
    fragment_attributes/
        segment_id/   207 chunk files,    57.5 KiB
    links/
        0/
            +1.+1.0/     3 chunk files,     0.1 KiB
            +1.-1.0/     9 chunk files,     0.4 KiB
            +1.0.+1/     4 chunk files,     0.2 KiB
            +1.0.-1/     2 chunk files,     0.1 KiB
            +1.0.0/   138 chunk files,    42.2 KiB
            0.+1.+1/     5 chunk files,     0.2 KiB
            0.+1.-1/     8 chunk files,     0.4 KiB
            0.+1.0/   136 chunk files,    43.9 KiB
            0.0.+1/   132 chunk files,    43.5 KiB
    object_attributes/
        length/     1 chunk files,   256.0 KiB
        vertex_count/     1 chunk files,     2.7 KiB
        weight/     1 chunk files,   256.0 KiB
    object_index/
        manifests/     1 chunk files,    33.8 KiB
        object_ids/     1 chunk files,    15.6 KiB
    vertex_attributes/
        fa/   207 chunk files,   872.4 KiB
    vertex_fragments/   207 chunk files,   119.6 KiB
    vertices/   

### 4c · Object attributes per level

Object attributes are kept for the streamlines each level keeps.

In [7]:
from zarr_vectors.building import get_resolution_level, open_store, read_object_attributes

root = open_store(STORE)
for lv in ds.levels:
    level_group = get_resolution_level(root, lv)
    kept = ds.level(lv).objects.ids()
    length = read_object_attributes(level_group, "length")[kept]
    print(f"Level {lv}: {len(kept):>5,} streamlines, mean length {length.mean():5.1f} mm")

Level 0: 2,000 streamlines, mean length  55.2 mm
Level 1: 1,000 streamlines, mean length  55.2 mm
Level 2:   500 streamlines, mean length  54.5 mm
Level 3:   250 streamlines, mean length  55.6 mm


## 5 · Validate, read back, export

### 5a · Validate

Level 5 adds the pyramid checks.

In [8]:
from zarr_vectors.validate import validate

print(validate(STORE, level=5).summary())

Level 5 validation: PASS
  71 passed, 0 warnings, 0 errors


### 5b · One streamline at every level

Object ids are the same at every level, so one streamline can be followed up
the pyramid while the sparsity keeps it. The tolerances compound: level *n*
may stray from the original by at most their sum up to *n* (0.5, 1.5 and
3.5 mm).

In [9]:
from zarr_vectors.types.polylines import read_polylines

def max_deviation(points, path):
    """Largest distance from any of `points` to the polyline `path`."""
    a, b = path[:-1], path[1:]
    ab = b - a
    t = np.einsum("psd,sd->ps", points[:, None, :] - a[None], ab) / np.einsum("sd,sd->s", ab, ab)
    nearest = a[None] + np.clip(t, 0, 1)[..., None] * ab[None]
    return np.linalg.norm(points[:, None, :] - nearest, axis=2).min(axis=1).max()

oid = int(ds.level(ds.levels[-1]).objects.ids()[0])   # kept at every level
original = streamlines[oid]
for lv in ds.levels:
    r = read_polylines(STORE, level=lv, object_ids=[oid])
    path = np.concatenate(r["polylines"][0]).astype(np.float64)
    print(f"Level {lv}: streamline {oid} has {len(path):>3} vertices, "
          f"max deviation from the original {max_deviation(original, path):.2f} mm")

Level 0: streamline 18 has 125 vertices, max deviation from the original 0.00 mm
Level 1: streamline 18 has  13 vertices, max deviation from the original 0.48 mm
Level 2: streamline 18 has  12 vertices, max deviation from the original 0.53 mm
Level 3: streamline 18 has   7 vertices, max deviation from the original 1.53 mm


### 5c · Export a coarse level to TRK

`export_trk` writes the store's TRK header back, so the exported file uses
the original reference volume.

In [10]:
from zarr_vectors_tools.convert.export.trk import export_trk

out_path = os.path.join(_tmpdir, "tracts_level2.trk")
export_trk(STORE, out_path, level=2, attribute_names=["fa"], object_attribute_names=["weight"])

trk = nib.streamlines.load(out_path)
print(f"streamlines : {len(trk.streamlines):,}")
print(f"points      : {sum(len(s) for s in trk.streamlines):,}")
print(f"dimensions  : {trk.header['dimensions']}, voxel size {trk.header['voxel_sizes']}")
print(f"scalars     : {list(trk.tractogram.data_per_point)}, properties: {list(trk.tractogram.data_per_streamline)}")
print(f"size        : {os.path.getsize(out_path) / 1e6:.2f} MB (input {os.path.getsize(TRK_PATH) / 1e6:.2f} MB)")

streamlines : 500
points      : 4,062
dimensions  : [ 90 108  90], voxel size [2. 2. 2.]
scalars     : ['fa'], properties: ['weight']
size        : 0.07 MB (input 3.58 MB)
